<div align="center">

# Projet de fin de module — Data Engineering

## Sujet 3 — Santé publique : suivi épidémiologique au CHU d'Abidjan

### Construction d'un entrepôt épidémiologique pour le CHU de Treichville

---

**Membres du binôme**

| Nom et prénoms 
|---|---|
| *[KOUAME Guy Marc Axel]* 

**Sujet :** n° 3 — Santé publique, suivi épidémiologique au CHU d'Abidjan
**Dépôt GitHub :** *[https://github.com/Vxell/projet-de-chu-treichville.git]*
**Enseignant :** GOUAH Tato Serge — Module Data Engineering

---
</div>

## Objet du notebook

Ce notebook exécute la totalité du pipeline, de la lecture du fichier brut issu du
système d'information hospitalier jusqu'au tableau de bord épidémiologique, en
passant par la pseudonymisation réglementaire, le chargement dans Supabase et la
modélisation en étoile.

Il s'exécute intégralement (`Kernel → Restart & Run All`) sans intervention
manuelle. Lorsque la connexion Supabase n'est pas configurée, les requêtes
analytiques sont rejouées sur un moteur SQL local afin que l'exécution reste
complète — le SQL exécuté est strictement identique.

## Sommaire

| Étape | Contenu |
|---|---|
| 0 | Configuration de l'environnement |
| 1 | Extraction et audit du fichier source |
| 2 | Nettoyage des données |
| 3 | Pseudonymisation  |
| 4 | Enrichissement métier |
| 5 | Contrôle qualité |
| 6 | Modélisation en étoile |
| 7 | Chargement dans Supabase |
| 8 | Analyses SQL |
| 9 | Tableau de bord |
| 10 | Synthèse et indicateurs clés |

> **Outils d'IA utilisés.**  :
> *[ Claude (Anthropic), pour la génération du jeu de données de départ et la relecture critique du code ».]*

---
# 0. Configuration de l'environnement

Le code métier est regroupé dans `src/` plutôt que recopié dans le notebook :
le DAG Airflow appelle exactement les mêmes fonctions, ce qui garantit que le
pipeline automatisé et le pipeline exploratoire produisent le même résultat.
Toute correction est ainsi appliquée aux deux à la fois.

In [1]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# --- Localisation de la racine du projet -------------------------------------
# Fonctionne aussi bien depuis notebooks/ (exécution locale) que depuis un
# environnement vierge type Google Colab, où le dépôt est cloné à la volée.
DEPOT = "https://github.com/Vxell/projet-de-chu-treichville.git"

racine = Path.cwd()
if not (racine / "src").exists():
    if (racine.parent / "src").exists():
        racine = racine.parent
    else:
        subprocess.run(["git", "clone", DEPOT, "projet"], check=True)
        racine = Path.cwd() / "projet"
os.chdir(racine)
sys.path.insert(0, str(racine / "src"))

print("Racine du projet :", racine)

Racine du projet : /Users/axelkouame/Desktop/projet-de-chu-treichville


In [2]:
# --- Chargement des secrets --------------------------------------------------
# Les identifiants ne figurent jamais dans le notebook : ils sont lus depuis
# un fichier .env exclu du dépôt Git.
try:
    from dotenv import load_dotenv
    load_dotenv(racine / ".env")
except ImportError:
    print("python-dotenv absent : les variables doivent être définies dans "
          "l'environnement.")

# Le sel de pseudonymisation doit rester constant d'une exécution à l'autre,
# sans quoi le même patient recevrait un pseudonyme différent à chaque
# chargement et le chaînage des séjours serait perdu. En son absence, on
# utilise une valeur de démonstration afin que le notebook reste exécutable,
# mais un déploiement réel exige un sel secret et stable.
if not os.environ.get("DE_SEL_PSEUDO"):
    os.environ["DE_SEL_PSEUDO"] = "sel_de_demonstration_a_remplacer"
    print("Sel de démonstration utilisé (voir .env.example pour la production).")

CHARGER_SUPABASE = bool(os.environ.get("SUPABASE_DB_URL"))
print("Chargement Supabase :", "activé" if CHARGER_SUPABASE else
      "désactivé (SUPABASE_DB_URL absente)")

Sel de démonstration utilisé (voir .env.example pour la production).
Chargement Supabase : désactivé (SUPABASE_DB_URL absente)


In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pipeline_etl as etl

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

FICHIER_SOURCE = racine / "data/raw/admissions_chu_treichville.csv"

# Le fichier source est généré si absent : le dépôt versionne le générateur
# (src/generate_dataset.py) et non les 25 Mo de CSV.
if not FICHIER_SOURCE.exists():
    print("Fichier source absent, génération en cours...")
    subprocess.run([sys.executable, "src/generate_dataset_chu.py"], check=True,
                   cwd=racine)
    Path("admissions_chu_treichville.csv").rename(FICHIER_SOURCE)

print("Fichier source :", FICHIER_SOURCE.name,
      f"({FICHIER_SOURCE.stat().st_size / 1e6:.1f} Mo)")

ModuleNotFoundError: No module named 'pipeline_etl'

---
# 1. Extraction et audit du fichier source

## 1.1 Contexte

Le fichier provient d'une extraction du système d'information hospitalier du CHU
de Treichville, couvrant deux années d'admissions (2023-2024). Il s'agit d'un
export brut : aucun traitement n'a été appliqué en amont, ni sur la qualité, ni
sur la confidentialité.

Deux conséquences pour le pipeline :

1. Le fichier contient des **données directement identifiantes** (nom, prénom,
   téléphone, date de naissance) qui ne peuvent pas entrer dans un entrepôt
   analytique en l'état.
2. Il porte les **défauts de saisie** habituels d'un SIH alimenté par plusieurs
   dizaines d'agents : codifications divergentes, doublons, valeurs aberrantes,
   dossiers incomplets.

L'audit ci-dessous en dresse l'inventaire chiffré.

In [ ]:
df_brut = etl.extraire(FICHIER_SOURCE)
df_brut.head(3)

In [ ]:
# Indicateurs de synthèse : ils ouvrent la section 3 du rapport
resume = etl.resumer_audit(df_brut)
pd.Series(resume).to_frame("valeur")

In [ ]:
# Audit détaillé colonne par colonne
audit = etl.auditer(df_brut)
audit

## 1.2 Lecture de l'audit

Trois familles de problèmes ressortent.

**Complétude.** Plusieurs colonnes présentent des valeurs manquantes : la durée
de séjour (dossiers clos sans date de sortie), l'identifiant du médecin (séjours
non attribués), la température d'entrée (constante non relevée) et la couverture
maladie.

**Cardinalité anormale.** La colonne `pathologie` affiche un nombre de modalités
distinctes très supérieur aux 31 diagnostics du référentiel médical. L'écart
provient uniquement de variations de casse et d'espaces parasites. La colonne
`sexe` présente le même symptôme.

**Doublons.** Des lignes strictement identiques figurent dans l'export, signe
d'une double soumission du même formulaire dans le SIH.

Vérifions ce diagnostic sur les colonnes concernées.

In [ ]:
print("Modalités de la colonne sexe :")
print(df_brut["sexe"].value_counts(dropna=False).to_string(), "\n")

print(f"Modalités distinctes de pathologie : {df_brut['pathologie'].nunique()} "
      f"(référentiel médical : 31)")
print("Exemples de variantes de la même pathologie :")
variantes = [v for v in df_brut["pathologie"].dropna().unique()
             if v.strip().lower() == "paludisme simple"]
print(" ", variantes, "\n")

print(f"Lignes strictement dupliquées : {df_brut.duplicated().sum():,}")